<a href="https://colab.research.google.com/github/hilmaniskandar06/Data-dari-google-Colab/blob/main/Airbnb_Tokyo_Market_Segmentation_%26_Clustering_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

# Membaca file dataset Airbnb yang terkompresi
# pandas otomatis bisa mengekstrak file .gz saat dibaca
df_listings = pd.read_csv('listings.csv.gz', compression='gzip', low_memory=False)

# Menampilkan informasi total baris dan kolom
print("Total Data:", df_listings.shape)

# Menampilkan 5 baris pertama untuk mengintip isi tabel
df_listings.head()

Total Data: (34419, 90)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,33121264,https://www.airbnb.com/rooms/33121264,20260630053204,2026-06-30,city scrape,新宿20m.新築build by Muji 駅徒歩8分Walmart 無料駐車四季bbq可,NaN,NaN,https://a0.muscache.com/pictures/99b5b45d-7f27...,116761693,...,4.84,4.59,4.56,M130014385,NaN,5,5,0,0,2.65
1,1707828556158854666,https://www.airbnb.com/rooms/1707828556158854666,20260630053204,2026-06-30,city scrape,駅徒歩3分｜ロフト付きコンパクトルーム｜国分寺10分｜中長期滞在歓迎（２階）,Relax and stretch your wings in a restful stay...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,474979321,...,NaN,NaN,NaN,M130063043,NaN,9,8,1,0,NaN
2,1428729810381657893,https://www.airbnb.com/rooms/1428729810381657893,20260630053204,2026-07-02,city scrape,ダブルルーム黄色 共有バスルーム,The room has a yellow tone.<br />You can see t...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,697092506,...,NaN,NaN,NaN,Hotels and Inns Business Act | 墨田区保健所 | 6墨福衛生環...,NaN,8,0,8,0,NaN
3,1671729464506095090,https://www.airbnb.com/rooms/1671729464506095090,20260630053204,2026-07-01,city scrape,新宿/西新宿5丁目駅徒歩3分/新宿駅まで3分/Switch/pocketWi-Fi/NET①,"Kick back and relax in this calm, stylish space.",NaN,https://a0.muscache.com/pictures/hosting/Hosti...,274793900,...,NaN,NaN,NaN,M130050822,NaN,33,30,3,0,NaN
4,1604694972630799987,https://www.airbnb.com/rooms/1604694972630799987,20260630053204,2026-07-01,city scrape,烟雨轩105 4号床（仅限女性预定）,Take it easy at this unique and tranquil getaway.,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,572381822,...,4.33,4.67,4.83,M130039910,NaN,8,0,1,7,1.36


In [2]:
# 1. Memilih kolom (variabel) yang berdampak pada nilai bisnis
kolom_pilihan = [
    'id', 'latitude', 'longitude', 'price',
    'minimum_nights', 'number_of_reviews',
    'review_scores_rating', 'availability_365'
]

# Membuat tabel baru yang lebih ramping
df_bersih = df_listings[kolom_pilihan].copy()

# 2. Membersihkan kolom 'price'
# Mengubah format string "$1,200.00" menjadi angka desimal 1200.0
df_bersih['price'] = df_bersih['price'].astype(str).str.replace('[\$,]', '', regex=True)
df_bersih['price'] = pd.to_numeric(df_bersih['price'], errors='coerce')

# 3. Menangani Data Kosong (Missing Values)
# Jika ada properti yang belum punya skor ulasan, kita isi dengan nilai tengah (median)
median_skor = df_bersih['review_scores_rating'].median()
df_bersih['review_scores_rating'] = df_bersih['review_scores_rating'].fillna(median_skor)

# Hapus baris data jika harganya kosong atau error
df_bersih = df_bersih.dropna(subset=['price'])

# Menampilkan hasil pembersihan
print("Total Data Siap Olah:", df_bersih.shape)
df_bersih.head()

<>:13: SyntaxWarning: invalid escape sequence '\$'
<>:13: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_7684/2215231096.py:13: SyntaxWarning: invalid escape sequence '\$'
  df_bersih['price'] = df_bersih['price'].astype(str).str.replace('[\$,]', '', regex=True)


Total Data Siap Olah: (32361, 8)


,id,latitude,longitude,price,minimum_nights,number_of_reviews,review_scores_rating,availability_365
0,33121264,35.727220,139.543910,49000.0,2.0,234,4.83,65
1,1707828556158854666,35.731860,139.474940,9647.5,4.0,0,4.83,364
2,1428729810381657893,35.706956,139.815976,5500.0,1.0,0,4.83,270
3,1671729464506095090,35.691490,139.683886,19100.0,1.0,0,4.83,340
4,1604694972630799987,35.752060,139.712938,4416.0,1.0,6,4.33,358


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 1. Menentukan kolom yang akan dipelajari oleh algoritma
# (Kita tidak memasukkan 'id', 'latitude', dan 'longitude' karena itu identitas/lokasi, bukan performa bisnis)
fitur_model = ['price', 'minimum_nights', 'number_of_reviews', 'review_scores_rating', 'availability_365']
X = df_bersih[fitur_model]

# 2. Standardisasi Skala Data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Menjalankan Algoritma K-Means
# Kita perintahkan mesin untuk membagi puluhan ribu properti ini menjadi 4 kelompok bisnis utama
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_bersih['cluster'] = kmeans.fit_predict(X_scaled)

# 4. Menyimpan Hasil Akhir untuk Dashboard Web (Fase 3 & 4)
nama_file_ekspor = 'airbnb_clustered.csv'
df_bersih.to_csv(nama_file_ekspor, index=False)

print(f"Selesai! Data berhasil dikelompokkan menjadi 4 klaster (0, 1, 2, 3).")
print(f"Tersimpan dengan nama: {nama_file_ekspor}")

# Menampilkan cuplikan hasilnya
df_bersih[['id', 'price', 'review_scores_rating', 'cluster']].head(10)


ValueError: Input X contains NaN.
KMeans does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 1. Menentukan kolom yang akan dipelajari oleh algoritma
fitur_model = ['price', 'minimum_nights', 'number_of_reviews', 'review_scores_rating', 'availability_365']

# PERBAIKAN: Hapus semua baris yang masih memiliki nilai kosong (NaN) di kelima kolom ini
df_bersih = df_bersih.dropna(subset=fitur_model)

X = df_bersih[fitur_model]

# 2. Standardisasi Skala Data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Menjalankan Algoritma K-Means
# Kita perintahkan mesin untuk membagi data ini menjadi 4 kelompok bisnis utama
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_bersih['cluster'] = kmeans.fit_predict(X_scaled)

# 4. Menyimpan Hasil Akhir untuk Dashboard Web (Fase 3 & 4)
nama_file_ekspor = 'airbnb_clustered.csv'
df_bersih.to_csv(nama_file_ekspor, index=False)

print(f"Selesai! Data berhasil dikelompokkan menjadi 4 klaster (0, 1, 2, 3).")
print(f"Tersimpan dengan nama: {nama_file_ekspor}")

# Menampilkan cuplikan hasilnya
df_bersih[['id', 'price', 'review_scores_rating', 'cluster']].head(10)


Selesai! Data berhasil dikelompokkan menjadi 4 klaster (0, 1, 2, 3).
Tersimpan dengan nama: airbnb_clustered.csv


,id,price,review_scores_rating,cluster
0,33121264,49000.00,4.83,0
1,1707828556158854666,9647.50,4.83,2
2,1428729810381657893,5500.00,4.83,2
3,1671729464506095090,19100.00,4.83,2
4,1604694972630799987,4416.00,4.33,2
6,48616140,28529.50,4.68,0
7,1572799152462582643,85875.00,4.97,2
8,1705932753344839068,35666.67,4.83,2
10,33147810,18400.00,4.79,2
11,1412744717556363935,24426.00,4.83,2
